# Turning weight-space programming on its head: quickstart

This notebook gets the repository, plays some of the published networks, checks the original experiment's
networks against their records, rebuilds the results tables from the saved reads, and trains a network.

In Colab, choose **Runtime > Change runtime type > GPU** first, then **Runtime > Run all**. It takes about five
minutes. Everything also runs without a GPU, more slowly.

It keeps the JAX that Colab already has, with its GPU support, and installs the versions of Flax, Optax and
Distrax that go with it. Scores therefore match the ones quoted here to within rounding, not to the last digit.
For the exact environment the networks were trained in, use the `Dockerfile` in the repository.

In [ ]:
print("Getting the code and the networks (about 30 seconds). The saved reads are fetched later, when they are needed.")
!git clone --depth 1 --filter=blob:none --sparse --progress https://github.com/letapjar/weight-space-verification.git
%cd weight-space-verification
!git sparse-checkout set code networks answer_key prompts scoring
%cd code

In [ ]:
# Keep the JAX this machine already has, with its GPU support, and install the Flax, Optax and Distrax that go
# with it. Without the constraint, pip upgrades JAX and leaves it disagreeing with Flax or with the GPU plugin.
from importlib.metadata import version
from pathlib import Path
kept = [f"{name}=={version(name)}" for name in ("jax", "jaxlib")]
Path("/tmp/keep_jax.txt").write_text(chr(10).join(kept))
print(f"Keeping JAX {version('jax')} and installing Flax, Optax and Distrax to match it (under a minute).")
print("pip may print a red dependency warning about packages this notebook does not use. Ignore it.")
!pip -q install -U flax optax distrax chex -c /tmp/keep_jax.txt
print("Done.")

In [ ]:
import flax, jax
print("JAX", jax.__version__, "and Flax", flax.__version__, "on", jax.devices()[0])

## Play published networks

Each file in `networks/` is one of the networks the models were asked to read, named by a random id. The answer
key says `93b4f9dc` is a Breakout network. It should score about 22.8.

In [ ]:
!python -m jaxport.play_network ../networks/93b4f9dc.npz breakout

The same network with its board turned 180°, and another Breakout network with all 500 of its inputs permuted.
Each plays its own version of the game as well as an unchanged network plays Breakout: about 22.8 and 22.6.

In [ ]:
!python -m jaxport.play_network ../networks/711c3b8f.npz breakout_turned180
!python -m jaxport.play_network ../networks/b585d2d8.npz breakout_shuffled1

A mixed network: Breakout's first layer on Pong's later layers. The models named it Breakout. It scores about
0.6, the level of random play.

In [ ]:
!python -m jaxport.play_network ../networks/5df3d4e0.npz breakout

## The original experiment's networks at our games

Our games were rebuilt from the original's description and recordings. Each line shows the original's network
playing our version of its game, beside the score in its own training record.

In [ ]:
!python -m jaxport.check_originals

## Rebuild the results tables from the saved reads

The first command fetches the saved reads, about 350 MB, which takes a minute or two and shows its progress.
The next reads every reply in `reads/` and rewrites `scoring/results.md` and `scoring/results.csv`. The last
line says whether the rebuilt tables are identical to the published ones.

In [ ]:
print("Fetching the saved reads and the weights files (a minute or two).")
!cd .. && git sparse-checkout add reads weights
!python results_table.py | tail -1
changed = !git status --short ../scoring
print("The rebuilt tables are identical to the published ones." if not changed else f"The rebuilt tables differ: {changed}")

## Train a network

The recipe of the twelve networks in the main table, cut from 200 million steps to 20 million. On a GPU this
takes under a minute and reaches a score of about 22, printing a line every few seconds. Without a GPU, change
`--steps` to `2000000`.

In [ ]:
!python -m jaxport.ppo --game breakout --activation tanh --seeds 0 --copies 256 --learning-rate 0.005 --steps 20000000 --runs-dir runs_quickstart

## The tests

The tests are written for the exact versions the networks were trained with, and one of them compares numbers
closely enough to depend on the JAX version. Run them in the repository's Docker image:
`docker build -t weight-space . && docker run --rm weight-space`.